## Begin ##

First load the dataset - fra_cleaned.csv.  df.shape prints the rows and columns of the dataset, and df.head displays the first five entries.

In [9]:
df = pd.read_csv('fra_cleaned.csv', encoding='latin-1', sep=';')
print(df.shape)
print(df.head())

(24063, 18)
                                                 url  \
0  https://www.fragrantica.com/perfume/xerjoff/ac...   
1  https://www.fragrantica.com/perfume/jean-paul-...   
2  https://www.fragrantica.com/perfume/jean-paul-...   
3  https://www.fragrantica.com/perfume/bruno-bana...   
4  https://www.fragrantica.com/perfume/jean-paul-...   

                          Perfume               Brand  Country  Gender  \
0  accento-overdose-pride-edition             xerjoff    Italy  unisex   
1            classique-pride-2024  jean-paul-gaultier   France   women   
2            classique-pride-2023  jean-paul-gaultier   France  unisex   
3               pride-edition-man        bruno-banani  Germany     men   
4         le-male-pride-collector  jean-paul-gaultier   France     men   

  Rating Value  Rating Count    Year  \
0         1,42           201  2022.0   
1         1,86            70  2024.0   
2         1,91           285  2023.0   
3         1,92            59  2019.0   
4     

## Get all notes ##

The next step is to build the fragrance matrix - First, we will combine all the notes from the top, middle, and base notes columns into one 'set' per fragrance.  This gives each fragrance a list like ['vanilla', 'amber', 'vetiver']

In [11]:
def get_all_notes(row):
    notes = []
    for col in ['Top', 'Middle', 'Base']:
        if pd.notna(row[col]):
            notes.extend([n.strip().lower() for n in row[col].split(',')])
    return notes

df['all_notes'] = df.apply(get_all_notes, axis=1)

print(df[['Perfume', 'all_notes']].head()) # Prints first 5 fragrances with note lists

                          Perfume  \
0  accento-overdose-pride-edition   
1            classique-pride-2024   
2            classique-pride-2023   
3               pride-edition-man   
4         le-male-pride-collector   

                                           all_notes  
0  [fruity notes, aldehydes, green notes, bulgari...  
1  [yuzu, citruses, orange blossom, neroli, musk,...  
2  [blood orange, yuzu, neroli, orange blossom, m...  
3  [guarana, grapefruit, red apple, walnut, laven...  
4  [mint, lavender, cardamom, artemisia, bergamot...  


## How many total notes? ##

This counts the unique notes in the dataset.

In [13]:
unique_notes = set()
for notes_list in df['all_notes']:
    unique_notes.update(notes_list)

print(f"Unique notes: {len(unique_notes)}")
print(f"\nFirst 10 (alphabetically): {sorted(unique_notes)[:10]}")

Unique notes: 1671

First 10 (alphabetically): ['absinthe', 'acai berry', 'accord eudora®', 'acerola', 'acerola blossom', 'acetylfuran', 'acácia', 'african freesia petals', 'african geranium', 'african ginger']


## Building the matrix ##

Messing around with populating the matrix using a dictionary data structure and numbers for column names vs just having the column name as the note.

TODO: bring up how creating a matrix at this step is a lot simpler than comparing string literals to each other

In [14]:
import numpy as np
import time

sorted_notes = sorted(unique_notes)

# --- O(n) approach: list scanning ---
start = time.time()

matrix_slow = np.zeros((len(df), len(sorted_notes)), dtype=int)

for row_idx, notes_list in enumerate(df['all_notes']):
    for note in notes_list:
        col_idx = sorted_notes.index(note)  # scans the list every time
        matrix_slow[row_idx, col_idx] = 1

slow_time = time.time() - start
print(f"List approach (O(n) lookup): {slow_time:.2f} seconds")

# --- O(1) approach: dictionary ---
start = time.time()

note_to_index = {note: i for i, note in enumerate(sorted_notes)}
matrix_fast = np.zeros((len(df), len(note_to_index)), dtype=int)

for row_idx, notes_list in enumerate(df['all_notes']):
    for note in notes_list:
        col_idx = note_to_index[note]  # direct hash lookup
        matrix_fast[row_idx, col_idx] = 1

fast_time = time.time() - start
print(f"Dict approach (O(1) lookup): {fast_time:.2f} seconds")

print(f"\nSpeedup: {slow_time / fast_time:.1f}x faster")
print(f"Matrices match: {np.array_equal(matrix_slow, matrix_fast)}")

List approach (O(n) lookup): 1.25 seconds
Dict approach (O(1) lookup): 0.05 seconds

Speedup: 25.0x faster
Matrices match: True


## O(n) vs. O(1) approaches ##

As we can see, the list approach is 25x faster - for a little over 24,000 fragrances, that gives us 1.25 seconds for the O(n) approach and 0.05 seconds for the O(1) approach!  The matrices are the exact same, but one was created 25x faster!

Though I only saved a little over a second with a dataset of this size, O(n) scales linearly with the number of features so it would only keep getting worse.

Let's make matrix_fast our note_matrix for the rest of the analysis and continue.

In [17]:
note_matrix = matrix_fast

print(f"Matrix shape: {note_matrix.shape}")
print(f"  {note_matrix.shape[0]} fragrances x {note_matrix.shape[1]} notes\n")

# How many notes does each fragrance have on average?
notes_per_frag = note_matrix.sum(axis=1)
print(f"Notes per fragrance - min: {notes_per_frag.min()}, max: {notes_per_frag.max()}, avg: {notes_per_frag.mean():.1f}\n")

# How sparse is the matrix? (mostly zeros)
total_cells = note_matrix.shape[0] * note_matrix.shape[1]
filled_cells = note_matrix.sum()
print(f"Sparsity: {filled_cells}/{total_cells} cells filled ({filled_cells/total_cells*100:.2f}%)\n")

# Spot check a known fragrance
idx = df[df['Perfume'].str.contains('shalimar', case=False)].index[0]
frag_notes = [note for note, col in note_to_index.items() if note_matrix[idx, col] == 1]
print(f"Spot check - {df.iloc[idx]['Perfume']}:")
print(f"  Notes: {frag_notes}")

Matrix shape: (24063, 1671)
  24063 fragrances x 1671 notes

Notes per fragrance - min: 1, max: 62, avg: 9.8

Sparsity: 236985/40209273 cells filled (0.59%)

Spot check - shalimar-souffle-d-oranger:
  Notes: ['bergamot', 'jasmine sambac', 'mandarin orange', 'neroli', 'orange blossom', 'petitgrain', 'sandalwood', 'vanilla']


In [ ]:
We got the matrix shape and a bit more info about our matrix. 24063 fragrances x 1671 notes.
The average fragrance has 9.8 notes out of a potential 1671!
This is a very sparse matrix.

In [21]:
search = df[df['Perfume'].str.contains('dakar', case=False)][['Perfume', 'Brand']]
print(search)

     Perfume             Brand
4119   dakar  thera-cosmeticos


In [22]:
target_idx = 4119
target_row = matrix_fast[target_idx]

# Jaccard: intersection / union for each fragrance
scores = []
for i in range(len(matrix_fast)):
    if i == target_idx:
        continue
    intersection = np.sum((target_row == 1) & (matrix_fast[i] == 1))
    union = np.sum((target_row == 1) | (matrix_fast[i] == 1))
    score = intersection / union if union > 0 else 0
    scores.append((i, score))

# Sort by score descending, grab top 10
scores.sort(key=lambda x: x[1], reverse=True)

print(f"Recommendations for: {df.iloc[target_idx]['Perfume']} by {df.iloc[target_idx]['Brand']}\n")
print(f"Its notes: {df.iloc[target_idx]['all_notes']}\n")
print("Top 10 Jaccard matches:")
for rank, (idx, score) in enumerate(scores[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — {score:.3f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

Recommendations for: dakar by thera-cosmeticos

Its notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao', 'woody notes', 'dried fruits']

Top 10 Jaccard matches:
  1. tobacco-vanille by tom-ford — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  2. sweetobacco by in-the-box — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woodsy notes', 'dried fruits']
  3. efrate by nuancielo — 0.778
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woodsy notes']
  4. vanille-persuasive by lpdo — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  5. tabac-gourmand by patrice-martin — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco bloss

## END PART 1 OF ANALYSIS ##

Jaccard performed well as a simple test, matching Fragrantica's similar fragrances for Dakar.  But it has some limitations: literal notes overlap, and fragrances with different numbers of notes may appear falsely dissimilar.

Enter NMF (this step takes a while but is only performed once - the decomposition of the matrix.

In [25]:
from sklearn.decomposition import NMF

# Fit NMF with 15 latent factors
nmf = NMF(n_components=15, random_state=42)
fragrance_factors = nmf.fit_transform(matrix_fast)  # 24063 x 15
factor_notes = nmf.components_                        # 15 x 1671

print(f"Fragrance x Factors: {fragrance_factors.shape}")
print(f"Factors x Notes: {factor_notes.shape}")
print(f"\nDakar's factor vector:")
print(fragrance_factors[target_idx].round(3))

Fragrance x Factors: (24063, 15)
Factors x Notes: (15, 1671)

Dakar's factor vector:
[0.    0.    0.127 0.003 0.    0.002 0.    0.    0.    0.    0.    0.115
 0.    0.    0.   ]


That vector is Dakar's identity compressed into 15 numbers. Each number says how much TV belongs to a particular archetype that the algorithm discovered.

Most of the values are 0 or near-zero — TV only loads heavily onto two factors: factor 2 (0.127) and factor 11 (0.115). That means NMF figured out that Dakar's character is mostly explained by two archetypes. Let's see what those archetypes actually represent:


In [24]:
sorted_notes = sorted(unique_notes)

for factor_idx in [2, 11]:
    top_notes = sorted(
        zip(sorted_notes, factor_notes[factor_idx]),
        key=lambda x: x[1],
        reverse=True
    )[:10]
    print(f"Factor {factor_idx} top notes:")
    for note, weight in top_notes:
        print(f"  {note}: {weight:.3f}")
    print()

Factor 2 top notes:
  vanilla: 7.849
  caramel: 0.430
  cinnamon: 0.380
  heliotrope: 0.375
  benzoin: 0.298
  plum: 0.244
  raspberry: 0.242
  apple: 0.235
  peach: 0.233
  almond: 0.230

Factor 11 top notes:
  tonka bean: 8.485
  benzoin: 1.073
  cinnamon: 0.767
  iris: 0.674
  tobacco: 0.415
  heliotrope: 0.365
  almond: 0.339
  pink pepper: 0.331
  labdanum: 0.316
  carnation: 0.228



The algorithm figured out Dakar's accords without being told anything about fragrance families explicitly!  Factor 2 is basically a gourmand vanilla archetype, and Factor 11 is more warm spicy with tonka bean and tobacco.  This is validated by Fragrantica's own accord pyramid.

That was fun, let's try cosine similarity

In [26]:
from numpy.linalg import norm

target_vector = fragrance_factors[target_idx]

scores_nmf = []
for i in range(len(fragrance_factors)):
    if i == target_idx:
        continue
    # Cosine similarity between two factor vectors
    dot = np.dot(target_vector, fragrance_factors[i])
    denom = norm(target_vector) * norm(fragrance_factors[i])
    score = dot / denom if denom > 0 else 0
    scores_nmf.append((i, score))

scores_nmf.sort(key=lambda x: x[1], reverse=True)

print(f"NMF recommendations for: {df.iloc[target_idx]['Perfume']} by {df.iloc[target_idx]['Brand']}\n")
print("Top 10:")
for rank, (idx, score) in enumerate(scores_nmf[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — {score:.3f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

NMF recommendations for: dakar by thera-cosmeticos

Top 10:
  1. tobacco-vanille by tom-ford — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  2. vanille-persuasive by lpdo — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  3. tabac-gourmand by patrice-martin — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  4. sweetobacco by in-the-box — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woodsy notes', 'dried fruits']
  5. efrate by nuancielo — 1.000
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woodsy notes']
  6. rem-eau-de-parfum by reminiscence — 1.000
     Notes: ['watery notes', 'floral notes',

Got some weird ones in here - a limitation of cosine similarity. Let's try euclidean distance instead

In [27]:
scores_euclidean = []
for i in range(len(fragrance_factors)):
    if i == target_idx:
        continue
    dist = np.sqrt(np.sum((target_vector - fragrance_factors[i]) ** 2))
    scores_euclidean.append((i, dist))

# Sort ascending — smaller distance = more similar
scores_euclidean.sort(key=lambda x: x[1])

print(f"NMF + Euclidean distance for: {df.iloc[target_idx]['Perfume']}\n")
print("Top 10:")
for rank, (idx, dist) in enumerate(scores_euclidean[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — dist: {dist:.4f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

NMF + Euclidean distance for: dakar

Top 10:
  1. tobacco-vanille by tom-ford — dist: 0.0000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  2. vanille-persuasive by lpdo — dist: 0.0019
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  3. tabac-gourmand by patrice-martin — dist: 0.0019
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  4. sweetobacco by in-the-box — dist: 0.0022
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woodsy notes', 'dried fruits']
  5. efrate by nuancielo — dist: 0.0022
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woodsy notes']
  6. chestnut-cream-french-vanilla by jousset-parfums — dist: 0.0039
  

Better, but we still got a weird one with Elixir Charnel Oriental Brulant.

Right now, every note in my matrix is weighted equally - must gets a 1 and oud gets a 1, even though musk appears in nearly half of all fragrances and oud in very few.  If two fragrances both contain oud, thats a stronger signal of similarity than two containing musk, right? Let's try weighting the note with TF-IDF frequency.

In [28]:
note_counts = matrix_fast.sum(axis=0)
sorted_notes_list = sorted(unique_notes)

most_common = sorted(zip(sorted_notes_list, note_counts), key=lambda x: x[1], reverse=True)[:10]
least_common = sorted(zip(sorted_notes_list, note_counts), key=lambda x: x[1])[:10]

print("Most common notes (will get LOW weight):")
for note, count in most_common:
    print(f"  {note}: appears in {count} fragrances")

print(f"\nLeast common notes (will get HIGH weight):")
for note, count in least_common:
    print(f"  {note}: appears in {count} fragrances")

Most common notes (will get LOW weight):
  musk: appears in 10951 fragrances
  bergamot: appears in 8612 fragrances
  sandalwood: appears in 8009 fragrances
  amber: appears in 7686 fragrances
  jasmine: appears in 7673 fragrances
  patchouli: appears in 7208 fragrances
  vanilla: appears in 6666 fragrances
  rose: appears in 6134 fragrances
  cedar: appears in 5582 fragrances
  mandarin orange: appears in 4006 fragrances

Least common notes (will get HIGH weight):
  acerola blossom: appears in 1 fragrances
  acetylfuran: appears in 1 fragrances
  african freesia petals: appears in 1 fragrances
  algerian geranium: appears in 1 fragrances
  alpinia: appears in 1 fragrances
  aluminum: appears in 1 fragrances
  alumroot: appears in 1 fragrances
  alyssum: appears in 1 fragrances
  angelica root: appears in 1 fragrances
  antillone: appears in 1 fragrances


In [29]:
import numpy as np

total_fragrances = matrix_fast.shape[0]
note_counts = matrix_fast.sum(axis=0)

# IDF weight for each note
idf = np.log(total_fragrances / (note_counts + 1))  # +1 to avoid division by zero

# Multiply: each 1 in the matrix gets replaced by its note's IDF weight
matrix_tfidf = matrix_fast * idf

print("Dakar's notes with their IDF weights:\n")
dakar_notes = [(sorted_notes_list[i], idf[i]) for i in range(len(idf)) if matrix_fast[target_idx, i] == 1]
dakar_notes.sort(key=lambda x: x[1], reverse=True)
for note, weight in dakar_notes:
    print(f"  {note}: {weight:.3f}")

Dakar's notes with their IDF weights:

  tobacco blossom: 6.238
  tobacco leaf: 5.556
  dried fruits: 5.379
  spicy notes: 4.583
  cacao: 4.422
  woody notes: 2.924
  tonka bean: 1.974
  vanilla: 1.284


Tobacco blossom is weighted highly, as it's less common, and vanilla is lower since it's in a good 1/4 of all fragrances

In [30]:
from numpy.linalg import norm

target_vector_tfidf = matrix_tfidf[target_idx]

scores_tfidf = []
for i in range(len(matrix_tfidf)):
    if i == target_idx:
        continue
    dot = np.dot(target_vector_tfidf, matrix_tfidf[i])
    denom = norm(target_vector_tfidf) * norm(matrix_tfidf[i])
    score = dot / denom if denom > 0 else 0
    scores_tfidf.append((i, score))

scores_tfidf.sort(key=lambda x: x[1], reverse=True)

print(f"TF-IDF + Cosine recommendations for: {df.iloc[target_idx]['Perfume']}\n")
print("Top 10:")
for rank, (idx, score) in enumerate(scores_tfidf[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — {score:.3f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

TF-IDF + Cosine recommendations for: dakar

Top 10:
  1. tobacco-vanille by tom-ford — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  2. sweetobacco by in-the-box — 0.938
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woodsy notes', 'dried fruits']
  3. efrate by nuancielo — 0.938
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woodsy notes']
  4. essences-44 by nuancielo — 0.899
     Notes: ['tobacco leaf', 'spicy notes', 'cacao', 'vanille', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woodsy notes']
  5. vanille-persuasive by lpdo — 0.846
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  6. tabac-gourmand by patrice-martin — 0.846
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka be

That was TF-IDF + cosine - cosine cares about the direction of the vectors while euclidean cares about magnitude too. So a fragrance with just vanilla and tonka could score high on cosine if those notes point in the same direction as Dakar, but Euclidean would penalize it for missing all that tobacco weight. Let's try TF-IDF + Euclidean!

In [31]:
scores_tfidf_euc = []
for i in range(len(matrix_tfidf)):
    if i == target_idx:
        continue
    dist = np.sqrt(np.sum((target_vector_tfidf - matrix_tfidf[i]) ** 2))
    scores_tfidf_euc.append((i, dist))

scores_tfidf_euc.sort(key=lambda x: x[1])

print(f"TF-IDF + Euclidean for: {df.iloc[target_idx]['Perfume']}\n")
print("Top 10:")
for rank, (idx, dist) in enumerate(scores_tfidf_euc[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — dist: {dist:.4f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

TF-IDF + Euclidean for: dakar

Top 10:
  1. tobacco-vanille by tom-ford — dist: 0.0000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  2. sweetobacco by in-the-box — dist: 4.3579
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woodsy notes', 'dried fruits']
  3. efrate by nuancielo — dist: 4.3579
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woodsy notes']
  4. essences-44 by nuancielo — dist: 5.6958
     Notes: ['tobacco leaf', 'spicy notes', 'cacao', 'vanille', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woodsy notes']
  5. vanille-persuasive by lpdo — dist: 6.9989
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  6. tabac-gourmand by patrice-martin — dist: 6.9989
     Notes: ['tobacco leaf', 'spicy

Interesting discovery - for TF-IDF, the weighting is doing a lot of the heavy lifting here so the top 10s were similar.